<a href="https://colab.research.google.com/github/florvela/IA-y-automatizacion-en-seguridad-defensiva/blob/main/codigos-de-ejemplo/05-integracion-conectores/05-integracion-conectores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Módulo 05 — Integración y Conectores

**Escenario:** El SOAR que construimos en el módulo anterior necesita hablar con el mundo real: el EDR que detecta malware, el firewall que bloquea IPs, Jira donde se crean los tickets, y VirusTotal para saber si un IOC es malicioso. Construimos la capa de conectores que hace todo eso posible.

El hilo conductor de este módulo es el **ConnectorRegistry**: lo creamos primero, vacío, y cada conector que definamos se registra ahí inmediatamente. Al final, el pipeline de respuesta solo habla con el registry — no sabe nada de cómo se implementó cada conector.

**Nota:** Todas las llamadas a APIs externas tienen modo `USE_MOCK = True` para correr el notebook sin credenciales reales.

Objetivos:
- Crear el registry como backbone del sistema
- Diseñar `ConectorBase` como contrato común
- Implementar y registrar conectores para VirusTotal, CrowdStrike y Jira
- Orquestar un pipeline de respuesta que solo usa el registry

In [1]:
# pip install requests
import os
import time
import json
import logging
import requests
from abc import ABC, abstractmethod
from datetime import datetime
from dataclasses import dataclass, field
from typing import Any, Optional

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('conectores')

# ─────────────────────────────────────────────
#  CONFIGURACIÓN — cambiar USE_MOCK a False
#  y completar las API keys para uso real
# ─────────────────────────────────────────────
USE_MOCK = True

VT_API_KEY         = os.environ.get('VT_API_KEY', 'TU_VT_API_KEY_AQUI')
CROWDSTRIKE_ID     = os.environ.get('CS_CLIENT_ID', 'TU_CS_ID_AQUI')
CROWDSTRIKE_SECRET = os.environ.get('CS_CLIENT_SECRET', 'TU_CS_SECRET_AQUI')
JIRA_URL           = os.environ.get('JIRA_URL', 'https://tu-org.atlassian.net')
JIRA_USER          = os.environ.get('JIRA_USER', 'analista@empresa.com')
JIRA_TOKEN         = os.environ.get('JIRA_TOKEN', 'TU_JIRA_TOKEN_AQUI')

print(f'Modo: {"MOCK (demo sin credenciales)" if USE_MOCK else "REAL (usando APIs reales)"}')

Modo: MOCK (demo sin credenciales)


## 1. El Registry — primero el contenedor

Antes de definir ningún conector, creamos el registry. Es el punto central del sistema: los conectores se registran acá cuando se inicializan, y los playbooks los buscan acá cuando los necesitan.

El registry no sabe nada de VirusTotal ni de CrowdStrike. Solo sabe que hay objetos con un nombre y que responden a `verificar_salud()`.

In [2]:
class ConnectorRegistry:
    """Registro centralizado de todos los conectores del SOAR."""

    def __init__(self):
        self._conectores = {}

    def registrar(self, nombre: str, conector):
        self._conectores[nombre] = conector
        logger.info(f'Conector registrado: {nombre}')

    def obtener(self, nombre: str):
        if nombre not in self._conectores:
            raise KeyError(f'Conector no registrado: {nombre}')
        return self._conectores[nombre]

    def verificar_todos(self) -> dict:
        """Health check de todos los conectores registrados."""
        return {nombre: c.verificar_salud() for nombre, c in self._conectores.items()}

    def listar(self) -> list:
        return list(self._conectores.keys())


# Creamos el registry ahora, vacío — los conectores se irán registrando
registry = ConnectorRegistry()
print('Registry creado. Conectores registrados:', registry.listar())

Registry creado. Conectores registrados: []


## 2. ConectorBase — el contrato que todos cumplen

Sin una clase base, cada conector es un mundo: distinto manejo de errores, distinto logging, distintos reintentos. Con `ConectorBase`, todos los conectores hablan el mismo idioma y el registry puede tratarlos de forma uniforme.

In [3]:
class ConectorBase(ABC):
    """Clase base para todos los conectores del SOAR."""

    def __init__(self, nombre: str, config: dict):
        self.nombre = nombre
        self.config = config
        self.use_mock = config.get('use_mock', True)
        self._logger = logging.getLogger(f'conector.{nombre}')

    @abstractmethod
    def verificar_salud(self) -> bool:
        """Verifica que el conector puede comunicarse con el servicio."""
        pass

    def _log(self, msg: str, nivel: str = 'info'):
        getattr(self._logger, nivel)(f'[{self.nombre}] {msg}')

    def _reintentar(self, fn, max_intentos: int = 3, backoff: float = 1.0):
        """Ejecuta una función con reintentos y backoff exponencial."""
        for intento in range(1, max_intentos + 1):
            try:
                return fn()
            except Exception as e:
                if intento == max_intentos:
                    self._log(f'Falló tras {max_intentos} intentos: {e}', 'error')
                    raise
                espera = backoff * (2 ** (intento - 1))
                self._log(f'Intento {intento} fallido. Reintentando en {espera}s...', 'warning')
                time.sleep(espera)

print('Clase base ConectorBase definida')

Clase base ConectorBase definida


## 3. Conector VirusTotal — lo definimos y lo registramos

VirusTotal agrega resultados de 70+ motores antivirus. La API gratuita permite 4 requests/minuto. En modo mock devolvemos respuestas realistas.

Después de definir el conector, lo registramos inmediatamente en el registry.

In [4]:
# Respuestas mock realistas para la demo
MOCK_VT = {
    'hash': {
        '44d88612fea8a8f36de82e1278abb02f': {  # Hash de EICAR test file
            'malicious': 58, 'suspicious': 3, 'undetected': 12,
            'tipo': 'Win32 EXE', 'nombre': 'EICAR-Test-File',
            'veredicto': 'MALICIOSO'
        },
        'd41d8cd98f00b204e9800998ecf8427e': {  # Hash de archivo vacío
            'malicious': 0, 'suspicious': 0, 'undetected': 71,
            'tipo': 'empty', 'nombre': '',
            'veredicto': 'LIMPIO'
        },
    },
    'ip': {
        '173.234.31.186': {
            'malicious': 12, 'pais': 'US', 'asn': 'AS14618 AMAZON-AES',
            'veredicto': 'MALICIOSA'
        },
        '45.33.32.156': {
            'malicious': 8, 'pais': 'US', 'asn': 'AS63949 LINODE-AP',
            'veredicto': 'SOSPECHOSA'
        },
    },
    'dominio': {
        'ns.marryaldkfaczcz.com': {
            'malicious': 22, 'categorias': ['malware', 'c2'],
            'veredicto': 'MALICIOSO'
        },
    }
}

class ConectorVirusTotal(ConectorBase):
    BASE_URL = 'https://www.virustotal.com/api/v3'

    def __init__(self, api_key: str, use_mock: bool = True):
        super().__init__('virustotal', {'use_mock': use_mock})
        self.api_key = api_key
        self.headers = {'x-apikey': api_key}

    def verificar_salud(self) -> bool:
        if self.use_mock:
            self._log('Health check OK (mock)')
            return True
        try:
            r = requests.get(f'{self.BASE_URL}/ip_addresses/8.8.8.8', headers=self.headers, timeout=5)
            return r.status_code == 200
        except Exception:
            return False

    def analizar_hash(self, hash_valor: str) -> dict:
        self._log(f'Analizando hash: {hash_valor[:16]}...')
        if self.use_mock:
            return MOCK_VT['hash'].get(hash_valor, {
                'malicious': 0, 'veredicto': 'NO_ENCONTRADO',
                'nota': 'Hash no visto antes — puede ser nuevo malware o archivo legítimo'
            })
        url = f'{self.BASE_URL}/files/{hash_valor}'
        def _llamar(): return requests.get(url, headers=self.headers, timeout=10).json()
        data = self._reintentar(_llamar)
        stats = data['data']['attributes']['last_analysis_stats']
        return {'malicious': stats['malicious'], 'veredicto': 'MALICIOSO' if stats['malicious'] > 0 else 'LIMPIO'}

    def analizar_ip(self, ip: str) -> dict:
        self._log(f'Analizando IP: {ip}')
        if self.use_mock:
            return MOCK_VT['ip'].get(ip, {'malicious': 0, 'veredicto': 'SIN_DATOS'})
        url = f'{self.BASE_URL}/ip_addresses/{ip}'
        def _llamar(): return requests.get(url, headers=self.headers, timeout=10).json()
        data = self._reintentar(_llamar)
        stats = data['data']['attributes']['last_analysis_stats']
        return {'malicious': stats['malicious'], 'veredicto': 'MALICIOSA' if stats['malicious'] > 5 else 'LIMPIA'}

    def analizar_dominio(self, dominio: str) -> dict:
        self._log(f'Analizando dominio: {dominio}')
        if self.use_mock:
            return MOCK_VT['dominio'].get(dominio, {'malicious': 0, 'veredicto': 'SIN_DATOS'})
        url = f'{self.BASE_URL}/domains/{dominio}'
        def _llamar(): return requests.get(url, headers=self.headers, timeout=10).json()
        data = self._reintentar(_llamar)
        stats = data['data']['attributes']['last_analysis_stats']
        return {'malicious': stats['malicious'], 'veredicto': 'MALICIOSO' if stats['malicious'] > 3 else 'LIMPIO'}


# Registramos en el registry
registry.registrar('virustotal', ConectorVirusTotal(VT_API_KEY, use_mock=USE_MOCK))

# Verificamos que funciona
vt = registry.obtener('virustotal')
resultado = vt.analizar_hash('44d88612fea8a8f36de82e1278abb02f')
print(f'Hash EICAR     → {resultado["veredicto"]} | {resultado.get("malicious", 0)}/73 detecciones')

resultado_ip = vt.analizar_ip('173.234.31.186')
print(f'IP sospechosa  → {resultado_ip["veredicto"]} | País: {resultado_ip.get("pais", "N/A")}')

print(f'\nRegistry: {registry.listar()}')

Hash EICAR     → MALICIOSO | 58/73 detecciones
IP sospechosa  → MALICIOSA | País: US

Registry: ['virustotal']


## 4. Conector CrowdStrike EDR — lo definimos y lo registramos

In [5]:
class ConectorCrowdStrike(ConectorBase):
    BASE_URL = 'https://api.crowdstrike.com'

    def __init__(self, client_id: str, client_secret: str, use_mock: bool = True):
        super().__init__('crowdstrike', {'use_mock': use_mock})
        self.client_id = client_id
        self.client_secret = client_secret
        self._token: Optional[str] = None

    def verificar_salud(self) -> bool:
        if self.use_mock:
            return True
        try:
            self._obtener_token()
            return self._token is not None
        except Exception:
            return False

    def _obtener_token(self):
        if self.use_mock:
            self._token = 'mock_token_cs_abc123'
            return
        resp = requests.post(f'{self.BASE_URL}/oauth2/token',
                             data={'client_id': self.client_id, 'client_secret': self.client_secret})
        self._token = resp.json()['access_token']

    def buscar_dispositivo(self, hostname: str) -> dict:
        self._log(f'Buscando dispositivo: {hostname}')
        if self.use_mock:
            return {
                'device_id': 'abc123def456',
                'hostname': hostname,
                'os': 'Windows 10',
                'ip_local': '10.0.1.45',
                'estado': 'normal',
                'ultimo_visto': datetime.now().isoformat()
            }
        if not self._token:
            self._obtener_token()
        headers = {'Authorization': f'Bearer {self._token}'}
        resp = requests.get(f'{self.BASE_URL}/devices/queries/devices/v1',
                            headers=headers, params={'filter': f'hostname:"{hostname}"'})
        return resp.json()

    def aislar_dispositivo(self, device_id: str) -> bool:
        self._log(f'Aislando dispositivo: {device_id}')
        if self.use_mock:
            time.sleep(0.3)
            self._log('Dispositivo aislado exitosamente (mock)')
            return True
        if not self._token:
            self._obtener_token()
        headers = {'Authorization': f'Bearer {self._token}'}
        resp = requests.post(f'{self.BASE_URL}/devices/entities/devices/actions/contain/v1',
                             headers=headers, params={'action_name': 'contain'},
                             json={'ids': [device_id]})
        return resp.status_code == 200


# Registramos en el registry
registry.registrar('crowdstrike', ConectorCrowdStrike(CROWDSTRIKE_ID, CROWDSTRIKE_SECRET, use_mock=USE_MOCK))

# Verificamos que funciona
cs = registry.obtener('crowdstrike')
dispositivo = cs.buscar_dispositivo('LAPTOP-FINANCE-01')
print(f'Dispositivo: {dispositivo["hostname"]} | OS: {dispositivo["os"]} | IP: {dispositivo["ip_local"]}')

print(f'\nRegistry: {registry.listar()}')

Dispositivo: LAPTOP-FINANCE-01 | OS: Windows 10 | IP: 10.0.1.45

Registry: ['virustotal', 'crowdstrike']


## 5. Conector Jira — lo definimos y lo registramos

In [6]:
class ConectorJira(ConectorBase):

    def __init__(self, url: str, user: str, token: str, use_mock: bool = True):
        super().__init__('jira', {'use_mock': use_mock})
        self.url = url
        self.auth = (user, token)
        self._ticket_counter = 1000

    def verificar_salud(self) -> bool:
        if self.use_mock:
            return True
        try:
            resp = requests.get(f'{self.url}/rest/api/3/myself', auth=self.auth, timeout=5)
            return resp.status_code == 200
        except Exception:
            return False

    def crear_ticket(self, titulo: str, descripcion: str, prioridad: str = 'Medium') -> dict:
        self._log(f'Creando ticket: {titulo[:50]}')
        if self.use_mock:
            self._ticket_counter += 1
            ticket_id = f'SOC-{self._ticket_counter}'
            return {
                'id': ticket_id,
                'url': f'{self.url}/browse/{ticket_id}',
                'estado': 'Open',
                'creado': datetime.now().isoformat()
            }
        payload = {
            'fields': {
                'project': {'key': 'SOC'},
                'summary': titulo,
                'description': {'type': 'doc', 'version': 1,
                                'content': [{'type': 'paragraph', 'content': [{'type': 'text', 'text': descripcion}]}]},
                'issuetype': {'name': 'Incident'},
                'priority': {'name': prioridad}
            }
        }
        resp = requests.post(f'{self.url}/rest/api/3/issue', json=payload, auth=self.auth)
        return resp.json()

    def actualizar_estado(self, ticket_id: str, comentario: str) -> bool:
        self._log(f'Actualizando {ticket_id}: {comentario[:50]}')
        if self.use_mock:
            return True
        payload = {'body': {'type': 'doc', 'version': 1,
                            'content': [{'type': 'paragraph', 'content': [{'type': 'text', 'text': comentario}]}]}}
        resp = requests.post(f'{self.url}/rest/api/3/issue/{ticket_id}/comment',
                             json=payload, auth=self.auth)
        return resp.status_code == 201


# Registramos en el registry
registry.registrar('jira', ConectorJira(JIRA_URL, JIRA_USER, JIRA_TOKEN, use_mock=USE_MOCK))

# Verificamos que funciona
jira = registry.obtener('jira')
ticket = jira.crear_ticket('[TEST] Conector verificado', 'Registro correcto en el registry.', 'Low')
print(f'Ticket de prueba: {ticket["id"]} → {ticket["url"]}')

print(f'\nRegistry: {registry.listar()}')

Ticket de prueba: SOC-1001 → https://tu-org.atlassian.net/browse/SOC-1001

Registry: ['virustotal', 'crowdstrike', 'jira']


## 6. Health check — verificamos que todo está en pie

Con los tres conectores registrados, verificamos el estado de todos a la vez antes de procesar cualquier alerta. Si algo falla acá, lo sabemos antes de que llegue la primera alerta real.

In [7]:
print('--- Health Check de conectores ---')
print(f'Conectores registrados: {registry.listar()}\n')

estado = registry.verificar_todos()
for nombre, ok in estado.items():
    icono = '✓' if ok else '✗'
    print(f'  {icono} {nombre}')

print('\nTodos los conectores operativos. Listo para procesar alertas.')

--- Health Check de conectores ---
Conectores registrados: ['virustotal', 'crowdstrike', 'jira']

  ✓ virustotal
  ✓ crowdstrike
  ✓ jira

Todos los conectores operativos. Listo para procesar alertas.


## 7. Pipeline de respuesta completo

Integrando todo: una alerta entra, se enriquece con VirusTotal, se aísla el endpoint con CrowdStrike, y se documenta en Jira. El pipeline solo habla con el registry — no importa qué conector está detrás.

In [8]:
def responder_a_malware(alerta: dict, registry: ConnectorRegistry) -> dict:
    """
    Pipeline completo de respuesta a detección de malware.
    Orquesta VirusTotal + CrowdStrike + Jira en un flujo automático.
    """
    vt  = registry.obtener('virustotal')
    cs  = registry.obtener('crowdstrike')
    jira = registry.obtener('jira')

    print(f'\n{"="*55}')
    print(f' RESPUESTA AUTOMÁTICA A MALWARE')
    print(f'{"="*55}')
    print(f' Hash:      {alerta["hash"]}')
    print(f' Endpoint:  {alerta["hostname"]}')
    print(f'{"="*55}')

    # 1. Enriquecimiento con VirusTotal
    print('\n[1/4] Consultando VirusTotal...')
    vt_resultado = vt.analizar_hash(alerta['hash'])
    es_malicioso = vt_resultado.get('veredicto') == 'MALICIOSO'
    print(f'      Veredicto: {vt_resultado["veredicto"]} | Detecciones: {vt_resultado.get("malicious",0)}')

    if not es_malicioso:
        print('\n→ Hash no confirmado como malicioso. Cerrando sin acción.')
        return {'resultado': 'falso_positivo'}

    # 2. Buscar dispositivo
    print('\n[2/4] Localizando endpoint en CrowdStrike...')
    dispositivo = cs.buscar_dispositivo(alerta['hostname'])
    print(f'      Encontrado: {dispositivo["hostname"]} | IP: {dispositivo["ip_local"]}')

    # 3. Aislar dispositivo
    print('\n[3/4] Aislando endpoint...')
    aislado = cs.aislar_dispositivo(dispositivo['device_id'])
    print(f'      Aislamiento: {"✓ EXITOSO" if aislado else "✗ FALLÓ"}')

    # 4. Crear ticket de incidente
    print('\n[4/4] Creando ticket en Jira...')
    detecciones = vt_resultado.get('malicious', 0)
    ticket = jira.crear_ticket(
        titulo=f'[CRÍTICO] Malware en {alerta["hostname"]} — {detecciones} detecciones VT',
        descripcion=(
            f'Hash: {alerta["hash"]}\n'
            f'Endpoint: {alerta["hostname"]} ({dispositivo["ip_local"]})\n'
            f'VirusTotal: {detecciones}/73 motores\n'
            f'Estado: dispositivo aislado automáticamente'
        ),
        prioridad='Highest'
    )
    print(f'      Ticket: {ticket["id"]} → {ticket["url"]}')

    print(f'\n→ Respuesta completada. Endpoint aislado. Ticket abierto.')
    return {'resultado': 'contenido', 'ticket': ticket['id'], 'aislado': aislado}


# Simular detección de malware en endpoint
alerta_malware = {
    'hash': '44d88612fea8a8f36de82e1278abb02f',
    'hostname': 'LAPTOP-FINANCE-01',
    'proceso': 'invoice_q4.exe',
    'ruta': 'C:\\Users\\jsmith\\Downloads\\invoice_q4.exe'
}
resultado = responder_a_malware(alerta_malware, registry)
print(f'\nResumen: {resultado}')


 RESPUESTA AUTOMÁTICA A MALWARE
 Hash:      44d88612fea8a8f36de82e1278abb02f
 Endpoint:  LAPTOP-FINANCE-01

[1/4] Consultando VirusTotal...
      Veredicto: MALICIOSO | Detecciones: 58

[2/4] Localizando endpoint en CrowdStrike...
      Encontrado: LAPTOP-FINANCE-01 | IP: 10.0.1.45

[3/4] Aislando endpoint...
      Aislamiento: ✓ EXITOSO

[4/4] Creando ticket en Jira...
      Ticket: SOC-1002 → https://tu-org.atlassian.net/browse/SOC-1002

→ Respuesta completada. Endpoint aislado. Ticket abierto.

Resumen: {'resultado': 'contenido', 'ticket': 'SOC-1002', 'aislado': True}
